In [1]:
import warnings
warnings.filterwarnings("ignore")
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()
import numpy as np
import pandas as pd



Instructions for updating:
non-resource variables are not supported in the long term


지문을 읽고 주제 분류하기

다음과 같은 데이터들을 데이터프레임으로 저장한다. 입력 데이터는 음식 및 스포츠 관련 지문으로 구성되어 있다.

In [2]:
paragraph_dict_list = [
    {'paragraph': 'Dishplace is located in sunnyvale downtown there is parking around the area but it can be difficult to find during peak business hours my sisters and i came to this place for dinner on a weekday they were really busy so i highly recommended making reservations unless you have the patience to wait', 'category': 'food'},
    {'paragraph': 'Srvice can be slower during busy hours but our waiter was courteous and help gave some great entree recommendations', 'category': 'food'},
    {'paragraph': 'Portions are huge both french toast and their various omelettes are really good their french toast is probably 1.5x more than other brunch places great place to visit if you are hungry and dont want to wait 1 hour for a table', 'category': 'food'},
    {'paragraph': 'We started with apps going the chicken and waffle slides and chicken nachos the sliders were amazing and the nachos were good too maybe by themselves the nachos would have scored better but after those sliders they were up against some tough competition', 'category': 'food'},
    {'paragraph': 'The biscuits and gravy was too salty two people in my group had the gravy and all thought it was too salty my hubby ordered a side of double egg and it was served on two small plates who serves eggs to one person on separate plates we commented on that when it was delivered and even the server laughed and said she doesnt know why the kitchen does that presentation of food is important and they really missed on this one', 'category': 'food'},
    {'paragraph': 'The garlic fries were a great starter (and a happy hour special) the pancakes looked and tasted great and were a fairly generous portion', 'category': 'food'},
    {'paragraph': 'Our meal was excellent i had the pasta ai formaggi which was so rich i didnt dare eat it all although i certainly wanted to excellent flavors with a great texture contrast between the soft pasta and the crisp bread crumbs too much sauce for me but a wonderful dish', 'category': 'food'},
    {'paragraph': 'What i enjoy most about palo alto is so many restaurants have dog-friendly seating outside i had bookmarked italico from when they first opened about a 1.5 years ago and was jonesing for some pasta so time to finally knock that bookmark off', 'category': 'food'},
    {'paragraph': 'The drinks came out fairly quickly a good two to three minutes after the orders were taken i expected my iced tea to taste a bit more sweet but this was straight up green tea with ice in it not to complain of course but i was pleasantly surprised', 'category': 'food'},
    {'paragraph': 'Despite the not so good burger the service was so slow the restaurant wasnt even half full and they took very long from the moment we got seated to the time we left it was almost 2 hours we thought that it would be quick since we ordered as soon as we sat down my coworkers did seem to enjoy their beef burgers for those who eat beef however i will not be returning it is too expensive and extremely slow service', 'category': 'food'},
    {'paragraph': 'The four reigning major champions simona halep caroline wozniacki angelique kerber and defending us open champion sloane stephens could make a case for being the quartet most likely to succeed especially as all but stephens has also enjoyed the no1 ranking within the last 14 months as they prepare for their gruelling new york campaigns they currently hold the top four places in the ranks', 'category': 'sports'},
    {'paragraph': 'The briton was seeded nn7 here last year before a slump in form and confidence took her down to no46 after five first-round losses but there have been signs of a turnaround including a victory over a sub-par serena williams in san jose plus wins against jelena ostapenko and victoria azarenka in montreal. konta pulled out of new haven this week with illness but will hope for good things where she first scored wins in a major before her big breakthroughs to the semis in australia and wimbledon', 'category': 'sports'},
    {'paragraph': 'Stephens surged her way back from injury in stunning style to win her first major here last year—and ranked just no83 she has since proved what a big time player she is winning the miami title via four fellow major champions then reaching the final at the french open back on north american hard courts she ran to the final in montreal only just edged out by halep she has also avoided many of the big names in her quarter—except for wild card azarenka as a possible in the third round', 'category': 'sports'},
    {'paragraph': 'When it came to england chances in the world cup it would be fair to say that most fans had never been more pessimistic than they were this year after enduring years of truly dismal performances at major tournaments – culminating in the 2014 event where they failed to win any of their three group games and finished in bottom spot those results led to the resignation of manager roy hodgson', 'category': 'sports'},
    {'paragraph': 'The team that eliminated russia – croatia – also improved enormously during the tournament before it began their odds were 33/1 but they played with real flair and star players like luka modric ivan rakitic and ivan perisic showed their quality on the world stage having displayed their potential by winning all three of their group stage games croatia went on to face difficult tests like the semi-final against england', 'category': 'sports'},
    {'paragraph': 'The perseyside outfit finished in fourth place in the premier league table and without a trophy last term after having reached the champions league final before losing to real madrid', 'category': 'sports'},
    {'paragraph': 'Liverpool fc will return to premier league action on saturday lunchtime when they travel to leicester city in the top flight as they look to make it four wins in a row in the league', 'category': 'sports'},
    {'paragraph': 'Alisson signed for liverpool fc from as roma this summer and the brazilian goalkeeper has helped the reds to keep three clean sheets in their first three premier league games', 'category': 'sports'},
    {'paragraph': 'But the rankings during that run-in to new york hid some very different undercurrents for murray had struggled with a hip injury since the clay swing and had not played a match since losing his quarter-final at wimbledon and he would pull out of the us open just two days before the tournament began—too late however to promote nederer to the no2 seeding', 'category': 'sports'},
    {'paragraph': 'Then came the oh-so-familiar djokovic-nadal no-quarter-given battle for dominance in the third set there were exhilarating rallies with both chasing to the net both retrieving what looked like winning shots nadal more than once pulled off a reverse smash and had his chance to seal the tie-break but it was djokovic serving at 10-9 who dragged one decisive error from nadal for a two-sets lead', 'category': 'sports'}
]

df = pd.DataFrame(paragraph_dict_list)
df

,paragraph,category
0,Dishplace is located in sunnyvale downtown the...,food
1,Srvice can be slower during busy hours but our...,food
2,Portions are huge both french toast and their ...,food
3,We started with apps going the chicken and waf...,food
4,The biscuits and gravy was too salty two peopl...,food
5,The garlic fries were a great starter (and a h...,food
6,Our meal was excellent i had the pasta ai form...,food
7,What i enjoy most about palo alto is so many r...,food
8,The drinks came out fairly quickly a good two ...,food
9,Despite the not so good burger the service was...,food


데이터 전처리

LSTM 모델이 입력 데이터를 처리할 수 있도록 수치값으로 변경한다.  
텍스트인 입력값을 수치로 변환하기 위해서 지문에 사용된 모든 단어들을 모아서 중복을 제거한 후 단어 리스트를 만든다.

In [3]:
# 중복을 제거해서 지문에 사용된 단어를 모으기 위해 빈 set을 선언한다.
words = set()
# print(type(words))

# 데이터프레임의 paragraph열(시리즈)을 모두 소문자로 변환한 후 공백을 경계로 나눠서 set에 저장한다.
# print(type(df.paragraph))
# print(type(df.paragraph.str))
# print(type(df.paragraph.str.lower()))
# print(type(df.paragraph.str.lower().str))
# print(type(df.paragraph.str.lower().str.split()))
df.paragraph.str.lower().str.split().apply(words.update)

# set에 저장된 단어들을 딕셔너리 변환화고 오름차순 정렬한다.
words = sorted(list(words))
print(len(words))
print(words)

537
['(and', '1', '1.5', '1.5x', '10-9', '14', '2', '2014', '33/1', 'a', 'about', 'action', 'after', 'against', 'ago', 'ai', 'alisson', 'all', 'almost', 'also', 'although', 'alto', 'amazing', 'american', 'and', 'angelique', 'any', 'apps', 'are', 'area', 'around', 'as', 'at', 'australia', 'avoided', 'azarenka', 'back', 'battle', 'be', 'beef', 'been', 'before', 'began', 'began—too', 'being', 'better', 'between', 'big', 'biscuits', 'bit', 'bookmark', 'bookmarked', 'both', 'bottom', 'brazilian', 'bread', 'breakthroughs', 'briton', 'brunch', 'burger', 'burgers', 'business', 'busy', 'but', 'by', 'came', 'campaigns', 'can', 'card', 'caroline', 'case', 'certainly', 'champion', 'champions', 'chance', 'chances', 'chasing', 'chicken', 'city', 'clay', 'clean', 'commented', 'competition', 'complain', 'confidence', 'contrast', 'could', 'course', 'courteous', 'courts', 'coworkers', 'crisp', 'croatia', 'crumbs', 'culminating', 'cup', 'currently', 'dare', 'days', 'decisive', 'defending', 'delivered', '

딕셔너리에 저장된 데이터를 sorted() 메소드를 사용해서 items() 메소드 실행 결과 생성되는 튜플의 인덱스를 지정해서 정렬할 수 있다.  
`import operator`  
`print(sorted(word2index.items(), key=operator.itemgetter(1)))`

set에 저장된 단어들을 아래와 같은 형태의 딕셔너리로 만든다.  
{0: 'more', 1: 'australia', ..., 535: 'this', 536: 'enormously'}

In [4]:
# enumerate() 함수는 인수로 지정한 객체에 저장된 내용을 (인덱스, 데이터) 형태로 리턴한다.
# for index, word in enumerate(words):
#     print(index, word)
index2word = dict(enumerate(words))
# print(type(index2word))
print(index2word)

{0: '(and', 1: '1', 2: '1.5', 3: '1.5x', 4: '10-9', 5: '14', 6: '2', 7: '2014', 8: '33/1', 9: 'a', 10: 'about', 11: 'action', 12: 'after', 13: 'against', 14: 'ago', 15: 'ai', 16: 'alisson', 17: 'all', 18: 'almost', 19: 'also', 20: 'although', 21: 'alto', 22: 'amazing', 23: 'american', 24: 'and', 25: 'angelique', 26: 'any', 27: 'apps', 28: 'are', 29: 'area', 30: 'around', 31: 'as', 32: 'at', 33: 'australia', 34: 'avoided', 35: 'azarenka', 36: 'back', 37: 'battle', 38: 'be', 39: 'beef', 40: 'been', 41: 'before', 42: 'began', 43: 'began—too', 44: 'being', 45: 'better', 46: 'between', 47: 'big', 48: 'biscuits', 49: 'bit', 50: 'bookmark', 51: 'bookmarked', 52: 'both', 53: 'bottom', 54: 'brazilian', 55: 'bread', 56: 'breakthroughs', 57: 'briton', 58: 'brunch', 59: 'burger', 60: 'burgers', 61: 'business', 62: 'busy', 63: 'but', 64: 'by', 65: 'came', 66: 'campaigns', 67: 'can', 68: 'card', 69: 'caroline', 70: 'case', 71: 'certainly', 72: 'champion', 73: 'champions', 74: 'chance', 75: 'chances'

{0: 'more', 1: 'australia', ..., 535: 'this', 536: 'enormously'} 형태의 딕셔너리를 {'more': 0, 'australia': 1, ..., 'this': 535, 'enormously': 536} 형태의 딕셔너리로 변환한다.

In [5]:
# word2index = {}
# for key in index2word.keys():
    # print(key, index2word[key])
    # word2index[index2word[key].strip()] = key
# for key, value in index2word.items():
    # print(key, value)
    # word2index[value.strip()] = key
    
word2index = {value.strip(): key for key, value in index2word.items()}
print(word2index)

{'(and': 0, '1': 1, '1.5': 2, '1.5x': 3, '10-9': 4, '14': 5, '2': 6, '2014': 7, '33/1': 8, 'a': 9, 'about': 10, 'action': 11, 'after': 12, 'against': 13, 'ago': 14, 'ai': 15, 'alisson': 16, 'all': 17, 'almost': 18, 'also': 19, 'although': 20, 'alto': 21, 'amazing': 22, 'american': 23, 'and': 24, 'angelique': 25, 'any': 26, 'apps': 27, 'are': 28, 'area': 29, 'around': 30, 'as': 31, 'at': 32, 'australia': 33, 'avoided': 34, 'azarenka': 35, 'back': 36, 'battle': 37, 'be': 38, 'beef': 39, 'been': 40, 'before': 41, 'began': 42, 'began—too': 43, 'being': 44, 'better': 45, 'between': 46, 'big': 47, 'biscuits': 48, 'bit': 49, 'bookmark': 50, 'bookmarked': 51, 'both': 52, 'bottom': 53, 'brazilian': 54, 'bread': 55, 'breakthroughs': 56, 'briton': 57, 'brunch': 58, 'burger': 59, 'burgers': 60, 'business': 61, 'busy': 62, 'but': 63, 'by': 64, 'came': 65, 'campaigns': 66, 'can': 67, 'card': 68, 'caroline': 69, 'case': 70, 'certainly': 71, 'champion': 72, 'champions': 73, 'chance': 74, 'chances': 75

word2index 딕셔너리를 활용해서 모든 지문을 구성하는 단어 목록을 수치로 변환한 파생 변수를 데이터프레임에 추가한다.

In [6]:
# Dishplace is located in sunnyvale downtown ... => 109 222 247 219 443 121 ...
print(word2index['dishplace'], word2index['is'], word2index['located'], word2index['in'], word2index['sunnyvale'], word2index['downtown'])

109 222 247 219 443 121


In [7]:
# 지문을 구성하는 모든 단어를 수치로 변환하는 함수를 만든다.
def encode_paragraph(paragraph):
    # print(type(paragraph))
    words = paragraph.split()
    # print(words)
    encode = []
    for word in words:
        encode.append([word2index[word.lower()]])
    # print(encode)
    return encode

In [8]:
# 지문이 수치로 변환된 결과를 데이터프레임에 파생 변수로 추가한다. => [[109], [222], [247], [219], [443], [121], [46...
df['encode_paragraph'] = df.paragraph.apply(encode_paragraph)
df

,paragraph,category,encode_paragraph
0,Dishplace is located in sunnyvale downtown the...,food,"[[109], [222], [247], [219], [443], [121], [46..."
1,Srvice can be slower during busy hours but our...,food,"[[430], [67], [38], [420], [124], [62], [207],..."
2,Portions are huge both french toast and their ...,food,"[[334], [28], [210], [52], [168], [475], [24],..."
3,We started with apps going the chicken and waf...,food,"[[506], [433], [524], [27], [177], [459], [77]..."
4,The biscuits and gravy was too salty two peopl...,food,"[[459], [48], [24], [180], [503], [476], [383]..."
5,The garlic fries were a great starter (and a h...,food,"[[459], [173], [169], [510], [9], [181], [434]..."
6,Our meal was excellent i had the pasta ai form...,food,"[[308], [264], [503], [140], [212], [185], [45..."
7,What i enjoy most about palo alto is so many r...,food,"[[511], [212], [132], [274], [10], [313], [21]..."
8,The drinks came out fairly quickly a good two ...,food,"[[459], [123], [65], [309], [148], [350], [9],..."
9,Despite the not so good burger the service was...,food,"[[102], [459], [292], [424], [178], [59], [459..."


분류 항목(food[1, 0], sports[0, 1])을 원-핫 인코딩으로 수치로 변환한 파생 변수를 데이터프레임에 추가한다.

In [9]:
# 분류 항목을 원-핫 인코딩으로 변환하는 함수를 만든다.
def encode_category(category):
    # print(category)
    if category == 'food':
        return [1, 0]
    else:
        return [0, 1]

In [10]:
# 분류 항목이 원-핫 인코딩으로 변환된 결과를 데이터프레임에 파생 변수로 저장한다.
df['encode_category'] = df.category.apply(encode_category)
df

,paragraph,category,encode_paragraph,encode_category
0,Dishplace is located in sunnyvale downtown the...,food,"[[109], [222], [247], [219], [443], [121], [46...","[1, 0]"
1,Srvice can be slower during busy hours but our...,food,"[[430], [67], [38], [420], [124], [62], [207],...","[1, 0]"
2,Portions are huge both french toast and their ...,food,"[[334], [28], [210], [52], [168], [475], [24],...","[1, 0]"
3,We started with apps going the chicken and waf...,food,"[[506], [433], [524], [27], [177], [459], [77]...","[1, 0]"
4,The biscuits and gravy was too salty two peopl...,food,"[[459], [48], [24], [180], [503], [476], [383]...","[1, 0]"
5,The garlic fries were a great starter (and a h...,food,"[[459], [173], [169], [510], [9], [181], [434]...","[1, 0]"
6,Our meal was excellent i had the pasta ai form...,food,"[[308], [264], [503], [140], [212], [185], [45...","[1, 0]"
7,What i enjoy most about palo alto is so many r...,food,"[[511], [212], [132], [274], [10], [313], [21]...","[1, 0]"
8,The drinks came out fairly quickly a good two ...,food,"[[459], [123], [65], [309], [148], [350], [9],...","[1, 0]"
9,Despite the not so good burger the service was...,food,"[[102], [459], [292], [424], [178], [59], [459...","[1, 0]"


각각의 입력값의 실제 길이(단어 개수)를 할 수 있도록 각 지문별 단어의 개수를 파생 변수로 데이터프레임에 추가한다.

In [11]:
# 지문을 구성하는 단어의 개수를 세는 함수를 만든다.
def words_count(paragraph):
    return len(paragraph.split())

In [12]:
# 지문을 구성하는 단어의 개수를 데이터프레임에 파생 변수로 추가한다.
df['words_count'] = df.paragraph.apply(words_count)
df

,paragraph,category,encode_paragraph,encode_category,words_count
0,Dishplace is located in sunnyvale downtown the...,food,"[[109], [222], [247], [219], [443], [121], [46...","[1, 0]",53
1,Srvice can be slower during busy hours but our...,food,"[[430], [67], [38], [420], [124], [62], [207],...","[1, 0]",19
2,Portions are huge both french toast and their ...,food,"[[334], [28], [210], [52], [168], [475], [24],...","[1, 0]",42
3,We started with apps going the chicken and waf...,food,"[[506], [433], [524], [27], [177], [459], [77]...","[1, 0]",43
4,The biscuits and gravy was too salty two peopl...,food,"[[459], [48], [24], [180], [503], [476], [383]...","[1, 0]",82
5,The garlic fries were a great starter (and a h...,food,"[[459], [173], [169], [510], [9], [181], [434]...","[1, 0]",24
6,Our meal was excellent i had the pasta ai form...,food,"[[308], [264], [503], [140], [212], [185], [45...","[1, 0]",50
7,What i enjoy most about palo alto is so many r...,food,"[[511], [212], [132], [274], [10], [313], [21]...","[1, 0]",43
8,The drinks came out fairly quickly a good two ...,food,"[[459], [123], [65], [309], [148], [350], [9],...","[1, 0]",49
9,Despite the not so good burger the service was...,food,"[[102], [459], [292], [424], [178], [59], [459...","[1, 0]",82


LSTM은 항상 같은 길이의 시퀀스를 받아야 한다.  
길이가 작은 입력 시퀀스는 패딩을 추가적으로 넣어서 모든 시퀀스 길이를 동일하게 설정한다.  
패딩이 LSTM 계산에 영향을 끼치지 않도록 패딩 이전의 입력 시퀀스의 실제 길이를 파라미터로 받아서 상태값 계산시 패딩을 제외한다.

In [13]:
# 가장 긴 지문의 단어 개수를 구한 후 모든 지문에 패딩을 넣어 가장 긴 지문과 동일한 길이를 갖도록 만들기 위해 가장 긴 지문의 단어 개수를 계산한다.
# max_words_count = 0
# for para in df.paragraph:
    # print(type(para), para)
    # if len(para.split()) > max_words_count:
        # max_words_count = len(para.split())
# for para in df.words_count:
    # if para > max_words_count:
        # max_words_count = para

max_words_count = df.words_count.max()
print(max_words_count)

91


In [14]:
# 모든 지문에 패딩([-1])을 집어넣어 가장 긴 문장과 동일한 길이를 갖게하는 함수를 만든다.
def sequence_padding(encode_paragraph):
    # print(type(encode_paragraph), len(encode_paragraph))
    for i in range(len(encode_paragraph), max_words_count):
        encode_paragraph.append([-1])
    return encode_paragraph

In [15]:
# 모든 지문에 패딩이 적용된 결과를 'encode_paragraph' 컬럼에 적용한다.
df['encode_paragraph'] = df.encode_paragraph.apply(sequence_padding)
print(len(df.encode_paragraph[1]))
print(df.encode_paragraph[1])
print(len(df.encode_paragraph[12]))
print(df.encode_paragraph[12])

91
[[430], [67], [38], [420], [124], [62], [207], [63], [308], [500], [503], [88], [24], [195], [174], [426], [181], [135], [362], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1], [-1]]
91
[[435], [444], [197], [505], [36], [170], [221], [219], [438], [439], [474], [521], [197], [156], [256], [198], [236], [533], [354], [229], [290], [406], [190], [414], [342], [511], [9], [47], [472], [329], [406], [222], [522], [459], [265], [473], [494], [166], [151], [256], [73], [462], [359], [459], [152], [32], [459], [168], [302], [36], [298], [291], [23], [189], [89], [406], [353], [474], [459], [152], [219], [271], [301], [

LSTM 모델에서 사용할 학습 데이터, 레이블, 지문의 실제 길이 파라미터를 만든다.

In [16]:
# numpy의 array() 메소드는 인수로 파이썬의 리스트나 튜플 데이터를 받아서 numpy 배열로 변환하는 기능을 실행하기 때문에 데이터프레임의 시리즈를 인수로 넣어주면
# 안되고 시리즈에 tolist() 메소드를 실행해서 파이썬의 리스트 형태로 변환시켜 넣어준다.
# print(type(df.encode_paragraph))
# print(type(df.encode_paragraph.tolist()))
x_train = np.array(df.encode_paragraph.tolist()) # LSTM 모델에 입력할 학습 데이터, 입력값
# print(type(x_train))
y_train = np.array(df.encode_category.tolist()) # LSTM 모델에 입력한 학습 데이터의 레이블, 실제값
words_count = np.array(df.words_count.tolist()) # 지문의 실제 길이

print(x_train.shape, y_train.shape, words_count.shape)

(20, 91, 1) (20, 2) (20,)


지문을 읽고 주제를 분류하는 LSTM 모델을 구현한다.

<img src="./LSTM3.png" wigth="1000" align="left" />

문맥 벡터(contextualized vector) 생성 단계

인덱스를 임베딩으로 변환한다.  
임베딩은 학습 과정을 통해 단어 유사도를 포함하게 되어 문맥 벡터를 생성하는데 도움을 준다.  
인간의 언어(자연어)는 수치화되어있지 않은 데이터이기 때문에 머신러닝, 딥러닝 바로 사용할 수 없다. 그래서 자연어 처리에서 특징을 추출해서 수치화를 해줘야 하는데 이때 사용하는 것이 `언어의 벡터화`이다. 이런 벡터화 과정을 `임베딩`이라고 한다.  
LSTM 모델에 임베딩된 시퀀스를 입력해서 최종 상태값을 출력한다. => 최종 상태값이 문맥 벡터이다.

주제(food, sports) 분류 단계

문맥 벡터를 `Dense(완전 연결, Fully Connected) Layer`에 입력하고 출력값을 노드가 2개인 Dense Layer에 입력한 후 노드가 2개인 Dense Layer의 출력값을 소프트 맥스에 입력해서 food, sports에 대한 예측값을 계산한다.

입력값과 실제값을 저장할 placeholder를 만든다.

In [17]:
# 입력값은 문장을 구성하는 단어들의 인덱스이며, 길이는 문장을 구성하는 단어의 최대 개수이다.
X = tf.placeholder(dtype=tf.float32, shape=[None, max_words_count, 1]) # 입력값을 기억할 placeholder, (20, 91, 1)
Y = tf.placeholder(dtype=tf.float32, shape=[None, 2]) # 실제값을 기억할 placeholder, (20, 2)

임베딩 및 LSTM 계층을 만든다.

In [18]:
# 임베딩 레이어는 입력값(단어들의 인덱스)을 입력받아 5차원의 벡터 임베딩을 출력하게 한다.
# layers.dense() 메소드는 Dense Layer 즉, 완전 연결 계층를 만든다.
# 입력 데이터 X를 입력받아 완전 연결 계층을 거치게 함으로써 5차원의 벡터 공간으로 임베딩(변한)한다. 입력값의 특성 차원을 신경망이 학습하기 좋은 5차원 형태로
# 확장, 변환해 주는 역할을 한다.
embedding = tf.layers.dense(X, 5)

# LSTM 셀은 64차원의 벡터의 생성값을 출력한다.
# LSTM 셀 내부의 은닉 상태(hidden state) 크기를 64차원으로 설정하여, 문장의 문맥 정보를 64개의 특성 벡터로 합축하고 기억할 수 있게 한다.
lstm_cell = tf.nn.rnn_cell.LSTMCell(num_units=64)

# LSTM 셀을 바탕으로 동적 RNN 연산을 수행해서 출력값과 상태값을 저장한다.
# sequence_length=words_count: 각 문장의 실제 길이(단어 개수)를 지정하여, 불필요한 패딩([-1]) 영역에 대한 불필요한 연산을 방지하고 정확한 계산을 돕는다.
# outputs: 모든 단어의 출력값을 기억한다.
# state: 문장의 마지막 시점에서의 전체 문장의 문맥 요약 정보를 기억한다.
outputs, state = tf.nn.dynamic_rnn(cell=lstm_cell, inputs=embedding, dtype=tf.float32, sequence_length=words_count)


Instructions for updating:
Please use `keras.layers.RNN(cell)`, which is equivalent to this API
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor


주제 분류를 위한 Dense Layer를 만든다.

In [19]:
# 주제 분류는 2개의 Dense Layer를 사용한다.
# 1번째 Dense Layer는 32개의 노드를 가지고 있고, 2번째 Dense Layer는 2개의 노드를 가지고 있으며 이 2개의 노드가 소프트 맥스의 입력으로 들어간다.
dense_layer = tf.layers.dense(state.h, 32)

# logit는 food, sports를 원-핫 인코딩으로 구분하기 위해서 2차원 벡터로 구성한다.
logit = tf.layers.dense(dense_layer, 2) # 최종 예측값

손실 함수 및 옵티마이저 정의

In [31]:
# 크로스 엔트로피 손실 함수로 모델의 예측값(logit)과 실제값(Y)를 비교해서 오차를 계산한다.
loss = tf.reduce_mean(
    tf.nn.softmax_cross_entropy_with_logits_v2(logits=logit, labels=Y)
)

# 모델이 예측 오차(loss)를 줄여나가도록 학습을 시키는 옵티마이저를 정의한다.
train = tf.train.AdamOptimizer(0.005).minimize(loss)

모델 요약

In [32]:
# 문맥 벡터 생성 단계
# 입력값은 단어들의 인덱스이며 그 길이는 항상 91이다.
print(X)
# 임베딩 레이어는 단어들의 인덱스를 받아 5차원 벡터의 임베딩으로 출력한다.
print(embedding)
# LSTM 셀은 64차원의 상태값을 출력한다.
print(state.h)

# 분류 단계
# 1번째 덴즈 레이어는 32개의 노드를 가진다.
print(dense_layer)
# 2번째 덴즈 레이어는 2개의 노드를 가지고 있으며, 소프트 맥스의 입력으로 들어간다.
print(logit)

Tensor("Placeholder:0", shape=(?, 91, 1), dtype=float32)
Tensor("dense/BiasAdd:0", shape=(?, 91, 5), dtype=float32)
Tensor("rnn/while/Exit_4:0", shape=(?, 64), dtype=float32)
Tensor("dense_1/BiasAdd:0", shape=(?, 32), dtype=float32)
Tensor("dense_2/BiasAdd:0", shape=(?, 2), dtype=float32)


학습 시킨다.

In [33]:
with tf.Session() as sess:
    sess.run(tf.global_variables_initializer())
    
    for epoch in range(201):
        _, _loss = sess.run([train, loss], feed_dict={X: x_train, Y: y_train})
        if epoch % 20 == 0:
            predict = tf.nn.softmax(logit) # 최종 예측값
            # 최종 예측값(predict)의 가장큰 값의 인덱스와 실제값(Y)의 가장큰 값의 인덱스를 비교한다. 결과는 True 또는 False 이다.
            correct_predict = tf.equal(tf.argmax(predict, 1), tf.argmax(Y, 1))
            # correct_predict에 저장된 True는 1로, False는 0으로 형변환 후 평균을 계산한다. 정확도를 계산하는 수식이다.
            accuracy = tf.reduce_mean(tf.cast(correct_predict, dtype=tf.float32))
            # 정확도를 계산한다.
            current_accuracy = accuracy.eval(feed_dict={X: x_train, Y: y_train})
            print('epoch: {:3d}, loss: {:7.5f}, accuracy: {:7.5f}'.format(epoch, _loss, current_accuracy))

epoch:   0, loss: 1.21704, accuracy: 0.50000
epoch:  20, loss: 0.62457, accuracy: 0.50000
epoch:  40, loss: 0.42211, accuracy: 0.80000
epoch:  60, loss: 0.27389, accuracy: 0.85000
epoch:  80, loss: 0.25709, accuracy: 0.95000
epoch: 100, loss: 0.08442, accuracy: 1.00000
epoch: 120, loss: 0.01767, accuracy: 1.00000
epoch: 140, loss: 0.07085, accuracy: 0.95000
epoch: 160, loss: 0.00907, accuracy: 1.00000
epoch: 180, loss: 0.00212, accuracy: 1.00000
epoch: 200, loss: 0.00131, accuracy: 1.00000
